
# USPS digits benchmark


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, NullFormatter, LogFormatterSciNotation

from src.kernel_pca import KernelPCA, RFF_KernelPCA
from src.usps_benchmarks import *

# automatic reload
%load_ext autoreload
%autoreload 2


In [ ]:
def timing_to_df(out_timing) -> pd.DataFrame:
    """
    Flatten a single run's timing dict into a tidy dataframe row.
    params: the hyperparameters for this run (n_components, n_rffs, etc.)
    """
    methods = ['ours', 'bakir', 'honeine', 'mika', 'kwok']
    params = {k: v for k, v in out_timing.items() if k not in methods}
    # kpca_ours_df = pd.DataFrame(out_timing['kpca_ours']).assign(method='kpca_ours')
    # kpca_bws_df = pd.DataFrame(out_timing['kpca_bws']).assign(method='kpca_bws')
    all_dfs = [pd.DataFrame(out_timing[method]).assign(method=method) for method in methods]
    out_timing_df = pd.concat(all_dfs, axis=0).assign(**params)
    return out_timing_df

In [ ]:
out_dir = "../../writeup/Kernel_PCA_Preimage_IEEE_TPAMI/"

## Benchmark metrics

In [ ]:
seed = 42
num_repeats = 10

#### effect of testing dataset size

In [ ]:
num_components = 50
num_train = 2000
num_test = [125, 250, 500, 1000, 2000]


all_metrics, all_timing = run_efficiency_benchmark(num_components=num_components, num_train=num_train, num_repeats=num_repeats, num_test=num_test)

In [ ]:
all_metrics_test_df = pd.concat(all_metrics, axis=0)
all_metrics_test_df.index = ['MSE', 'SSIM', 'PSNR'] * len(all_metrics)

all_timing_test_df = []
for out_timing in all_timing:
    all_timing_test_df.append(timing_to_df(out_timing))
all_timing_test_df = pd.concat(all_timing_test_df, axis=0)

# save output to CSV files
all_metrics_test_df.to_csv("results/usps_benchmark_efficiency_metrics_test_size.csv")
all_timing_test_df.to_csv("results/usps_benchmark_efficiency_time_and_memory_test_size.csv")

#### effect of number of components

In [ ]:
num_components = [25, 50, 100, 200]
num_train = 2000
num_test = 1000

all_metrics_comp, all_timing_comp = run_efficiency_benchmark(num_components=num_components, num_train=num_train, num_repeats=num_repeats, num_test=num_test)

In [ ]:
all_metrics_comp_df = pd.concat(all_metrics_comp, axis=0)
all_metrics_comp_df.index = ['MSE', 'SSIM', 'PSNR'] * len(all_metrics_comp)

all_timing_comp_df = []
for out_timing in all_timing_comp:
    all_timing_comp_df.append(timing_to_df(out_timing))
all_timing_comp_df = pd.concat(all_timing_comp_df, axis=0)

# save output to CSV files
all_metrics_comp_df.to_csv("results/usps_benchmark_efficiency_metrics_num_comp.csv")
all_timing_comp_df.to_csv("results/usps_benchmark_efficiency_time_and_memory_num_comp.csv")

**effect of training data size**

In [ ]:
num_components = 50
num_train = [500, 1000, 2000, 4000]
num_test = 1000

all_metrics_train, all_timing_train = run_efficiency_benchmark(num_components=num_components, num_train=num_train, num_repeats=num_repeats, num_test=num_test)

In [ ]:
all_metrics_train_df = pd.concat(all_metrics_train, axis=0)
all_metrics_train_df.index = ['MSE', 'SSIM', 'PSNR'] * len(all_metrics_train)

all_timing_train_df = []
for out_timing in all_timing_train:
    all_timing_train_df.append(timing_to_df(out_timing))
all_timing_train_df = pd.concat(all_timing_train_df, axis=0)

# save output to CSV files
all_metrics_train_df.to_csv("results/usps_benchmark_efficiency_metrics_train_size.csv")
all_timing_train_df.to_csv("results/usps_benchmark_efficiency_time_and_memory_train_size.csv")

### Visualization

In [214]:
def plot_efficiency_benchmark(df, variable, xlabel, out_file=None, logx=True,
     fontsize=8, colors=["black", "red", "blue", "purple", "orange"]):
    """
    Plot the efficiency benchmark results for a given measure.
    """    
    panels = {"time_s": "Elapsed time (s)", "peak_memory_mb": "Peak memory (MB)"}

    # --- prepare data once ---
    is_mem = df["measure"] == "peak_memory_kb"
    df.loc[is_mem, "total"] /= 1024
    df.loc[is_mem, "measure"] = "peak_memory_mb"

    df["log_total"] = np.log(df["total"])

    stats = (
        df.groupby(["method", "measure"] + [variable])["log_total"]
        .agg(mu="mean", sd="std")
        .reset_index()
        .fillna({"sd": 0})
    )
    stats["center"] = np.exp(stats["mu"])
    stats["lo"] = np.exp(stats["mu"] - stats["sd"])
    stats["hi"] = np.exp(stats["mu"] + stats["sd"])

    # --- plot ---
    #methods = df["method"].unique()
    methods = {'ours': 'Ours', 'bakir': 'B.W.S.', 'honeine': 'H.R.', 'mika': 'M. et al.', 'kwok': 'K.T.'}
    fig, axes = plt.subplots(1, 2, figsize=(4., 1.5), constrained_layout=True, sharex=True)

    for ax, (measure, ylabel) in zip(axes, panels.items()):
        for method, color in zip(methods.keys(), colors):
            d = stats[(stats["method"] == method) & (stats["measure"] == measure)]
            d = d.sort_values(variable)

            ax.errorbar(d[variable], d["center"], yerr=[d["center"] - d["lo"], d["hi"] - d["center"]],
                marker="o", ms=3.5, lw=1., color=color, capsize=1.5, elinewidth=0.5, label=methods[method])

        ax.set(xscale="log" if logx else "linear", yscale="log")
        ax.set_xlabel(xlabel, fontsize=fontsize)
        ax.set_ylabel(ylabel, fontsize=fontsize)
        ax.tick_params(axis="both", which="both", labelsize=fontsize)
        if logx:
            # fewer major ticks (one per decade, or every 2 decades)
            ax.xaxis.set_major_locator(LogLocator(base=10, numticks=5))
            # keep minor tick marks but hide their labels
            ax.xaxis.set_minor_formatter(NullFormatter())

    # single shared legend on top
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles, labels, loc="upper center", ncol=len(methods),
        bbox_to_anchor=(0.55, 1.15), fontsize=fontsize, frameon=False, columnspacing=0.5 
    )
    if out_file:
        fig.savefig(out_file, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()
    

**number of testing examples**

In [ ]:
all_timing_test_df = all_timing_test_df.reset_index().rename(columns={'index': 'measure'})

# replance NA values with 0 for fit_inverse and reconstruction
all_timing_test_df['fit_inverse'] = all_timing_test_df['fit_inverse'].fillna(0)
all_timing_test_df['reconstruction'] = all_timing_test_df['reconstruction'].fillna(0)
# total = fit_inverse + reconstruction
all_timing_test_df['total'] = all_timing_test_df['fit_inverse'] + all_timing_test_df['reconstruction']

outfile = out_dir + "figures/usps_benchmark_efficiency_time_and_memory_test_size.pdf"
plot_efficiency_benchmark(all_timing_test_df, 'n_test', 'Number of testing examples', out_file=outfile, logx=False, fontsize=7.5)

In [325]:
df = all_timing_test_df[["n_test", "method", "measure", "total", "seed"]]
df = df.set_index(["seed", "n_test", "measure"]).pivot(columns="method", values="total")
# calculate ratios
for method in df.columns:
    df[method] = df[method] / df['ours']
# aggregate
df.groupby(['n_test', 'measure']).mean().round(2)

method                  bakir  honeine     kwok     mika  ours
n_test measure                                                
125    peak_memory_mb   28.82    18.32    11.31     0.80   1.0
       time_s          185.92    84.16   110.99   236.73   1.0
250    peak_memory_mb   27.09    18.10    10.39     0.79   1.0
       time_s          204.49    83.57   234.64   529.25   1.0
500    peak_memory_mb   24.38    17.75     8.92     0.75   1.0
       time_s          193.68    73.24   438.77   990.95   1.0
1000   peak_memory_mb   20.74    17.28     6.96     0.70   1.0
       time_s          185.76    89.07   838.90  1906.61   1.0
2000   peak_memory_mb   16.79    16.78     4.83     0.65   1.0
       time_s          160.09    94.20  1418.81  3175.78   1.0

**number of training examples**

In [216]:
all_timing_train_df = all_timing_train_df.reset_index().rename(columns={'index': 'measure'})

# replance NA values with 0 for fit_inverse and reconstruction
all_timing_train_df['fit_inverse'] = all_timing_train_df['fit_inverse'].fillna(0)
all_timing_train_df['reconstruction'] = all_timing_train_df['reconstruction'].fillna(0)
# total = fit_inverse + reconstruction
all_timing_train_df['total'] = all_timing_train_df['fit_inverse'] + all_timing_train_df['reconstruction']

outfile = out_dir + "figures/usps_benchmark_efficiency_time_and_memory_train_size.pdf"
plot_efficiency_benchmark(all_timing_train_df, 'n_train', 'Number of training examples', out_file=outfile, logx=False, fontsize=7.5)

In [326]:
all_timing_train_df['seed'] = np.sort([i for i in range(1, 11)]*40)
df = all_timing_train_df[["n_train", "method", "measure", "total", "seed"]]
df = df.set_index(["seed", "n_train", "measure"]).pivot(columns="method", values="total")
# calculate ratios
for method in df.columns:
    df[method] = df[method] / df['ours']
# aggregate
df.groupby(['n_train', 'measure']).mean().round(2)

method                   bakir  honeine     kwok     mika  ours
n_train measure                                                
500     peak_memory_mb    3.27     4.56     1.12     0.78   1.0
        time_s           28.64    55.35  1865.96  1725.75   1.0
1000    peak_memory_mb    8.24     8.24     2.39     0.65   1.0
        time_s           79.72    85.44  1453.50  2093.46   1.0
2000    peak_memory_mb   20.73    17.28     6.96     0.70   1.0
        time_s          256.24   123.82  1240.38  2681.06   1.0
4000    peak_memory_mb   49.70    36.16    18.13     0.75   1.0
        time_s          845.31   209.75   838.96  2975.76   1.0

#### number of retained components

In [ ]:
all_timing_comp_df = all_timing_comp_df.reset_index().rename(columns={'index': 'measure'})

# replance NA values with 0 for fit_inverse and reconstruction
all_timing_comp_df['fit_inverse'] = all_timing_comp_df['fit_inverse'].fillna(0)
all_timing_comp_df['reconstruction'] = all_timing_comp_df['reconstruction'].fillna(0)
# total = fit_inverse + reconstruction
all_timing_comp_df['total'] = all_timing_comp_df['fit_inverse'] + all_timing_comp_df['reconstruction']

outfile = out_dir + "figures/usps_benchmark_efficiency_time_and_memory_comp_size.pdf"
plot_efficiency_benchmark(all_timing_comp_df, 'n_components', 'Number of components', out_file=outfile, logx=False, fontsize=7.5)